# 리딩방 → 건전 투자 유도형 필터링 파이프라인
### 텔레그램 크롤링 → 추천 종목 추출 → 언론 검증 → 기업 건전성 스코어링 → 거래량 급등 필터

원래는 "투자 리딩방 스팸을 이상치로 탐지"하는 프로젝트였지만, 방향을 넓혀 아래 흐름으로
**건전한 기업으로 투자금이 흘러가도록 돕는 필터링 시스템**을 만드는 것을 목표로 합니다.

**전체 흐름**
1. 텔레그램 리딩방 크롤링 (Telethon)
2. 리딩방 메시지에서 추천 종목/기업 추출
3. 추천 종목 관련 언론 기사 조사 (네이버 뉴스 검색 API)
4. 그 기사가 "작전세력성 보도"인지 "진짜 호재"인지 1차 판별 — **이상치 탐지 적용 지점 ①** (뉴스 버스트·근접중복 탐지)
5. 해당 기업의 재무/공시 자료로 안정성 스코어링
6. 거래량 급등 감지(**이상치 탐지 적용 지점 ②**) + 종합 판단으로 "건전 투자 유도" 북극성 지표 산출
7. 결과 저장 및 다운로드

> **이상치 탐지는 어디에 남아있나?**
> 프로젝트 전체가 하나의 이상치 탐지 모델은 아니지만, **4단계(뉴스 버스트 탐지)**와 **6단계(거래량 급등 탐지)**
> 두 지점에는 이상치 탐지 기법(근접중복 탐지 MinHash, 통계적 급등 탐지 + Isolation Forest)이 명시적으로 사용됩니다.

> **주의사항**
> - 텔레그램 크롤링은 본인 명의 API 자격증명으로 **공개 채널만** 대상으로 진행하세요.
> - 이 노트북은 기본적으로 `DEMO_MODE = True`로 설정되어 있어, 실제 API 자격증명 없이도
>   내장 샘플 데이터로 전체 파이프라인(1~7단계)을 바로 실행/테스트할 수 있습니다.
> - "작전세력 기사"라는 판별은 통계적 의심 신호일 뿐 법적 판단이 아닙니다. 최종 결론은 반드시 사람이 검토해야 합니다.
> - 3~5단계의 뉴스/기업 재무 데이터는 실제 API 자격증명이 없으면 데모 샘플 데이터로 대체되며, 실제 수치가 아닙니다.

> **회의 반영 (북극성 방향 확정)**: 사기 탐지형(A) vs 건전 투자 유도형(B) 중 **B로 방향을 확정**했습니다.
> 이에 따라 회의에서 지목된 세부 지표 3가지를 반영했습니다.
> 1. **언론사 신뢰도 스코어** (3-1단계) — "5만 원이면 네이버 뉴스에 기사를 뿌릴 수 있다"는 지적을 반영해 주요 언론사/소규모 인터넷 언론사를 구분
> 2. **뉴스-주가 상관관계** (3-2단계) — 뉴스 버스트가 실제 공시(실적·계약 등)와 맞물리는지 확인해 "숫자놀음"과 "진짜 호재"를 구분
> 3. **거래량 급등 탐지** (6단계, 기존 유지) — 소액으로도 흔들리는 소형주·바이오 기업의 취약성을 반영해 임계값을 동적으로 조정


In [ ]:
!pip install -q telethon datasketch openpyxl scikit-learn

---
## 0단계. 실행 모드 설정

`DEMO_MODE = True`  → 텔레그램/뉴스/재무 API 호출을 모두 건너뛰고 내장 샘플 데이터로 전체 파이프라인을 테스트
`DEMO_MODE = False` → 실제 Telethon 크롤링 + 실제 뉴스 API 호출 수행 (본인 API 자격증명 필요)


In [ ]:
DEMO_MODE = True  # 실제로 크롤링/뉴스 API를 호출하려면 False로 변경하세요


---
# 1단계. 텔레그램 리딩방 크롤링

## 1-1. 수집 (Telethon)

my.telegram.org 에서 발급받은 `api_id`/`api_hash`로 공개 채널 메시지를 수집합니다.


In [ ]:
import json, time, asyncio
from getpass import getpass

RAW_PATH = "telegram_raw.jsonl"

if not DEMO_MODE:
    # Colab/Jupyter 커널은 이미 asyncio 이벤트 루프를 돌리고 있어서,
    # telethon.sync의 동기식 `with client:` 문법은 "You must use async with ..." 에러가 납니다.
    # 따라서 진짜 async/await 문법으로 작성합니다.
    from telethon import TelegramClient

    api_id = int(getpass("api_id: "))
    api_hash = getpass("api_hash: ")
    client = TelegramClient("session_investment_spam", api_id, api_hash)

    async def crawl_channel(channel_username, out_path, limit=3000):
        entity = await client.get_entity(channel_username)
        with open(out_path, "a", encoding="utf-8") as f:
            async for msg in client.iter_messages(entity, limit=limit):
                if not msg.text:
                    continue
                record = {
                    "channel": channel_username,
                    "channel_id": entity.id,
                    "message_id": msg.id,
                    "date": msg.date.isoformat(),
                    "sender_id": msg.sender_id,
                    "text": msg.text,
                    "views": getattr(msg, "views", None),
                    "forwards": getattr(msg, "forwards", None),
                    "fwd_from": str(msg.fwd_from) if msg.fwd_from else None,
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        await asyncio.sleep(2)  # rate limit 완화
else:
    print("DEMO_MODE=True → 실제 크롤링은 건너뜁니다. (아래에서 샘플 데이터를 로드합니다)")


In [ ]:
# 실제 크롤링 시 아래 시드 채널 유저네임을 채워서 실행하세요.
SEED_CHANNELS = [
    "gogonero2",
    "aitodaystock",
    "YOUTUBE_INV1",
    "SEDOLSTOCK",
    "CryptoRich_02",
    "CryptoRich33",
    "CryptoRich37",
    "CryptoRich4",
    "CryptoRich43",
    "coinupkor00",
    "fpt_reviews"
]

if not DEMO_MODE:
    from telethon.errors import UsernameInvalidError, UsernameNotOccupiedError, FloodWaitError

    failed_channels = []

    async with client:
        for ch in SEED_CHANNELS:
            try:
                await crawl_channel(ch, RAW_PATH)
                print(f"  ✓ {ch} 수집 완료")
            except (UsernameInvalidError, UsernameNotOccupiedError):
                print(f"  ✗ {ch}: 존재하지 않거나 삭제된 채널 (건너뜀)")
                failed_channels.append(ch)
            except FloodWaitError as e:
                print(f"  ✗ {ch}: 요청 제한(FloodWait) {e.seconds}초 대기 필요 — 건너뜀")
                failed_channels.append(ch)
            except Exception as e:
                print(f"  ✗ {ch}: {type(e).__name__} — {e}")
                failed_channels.append(ch)

    print(f"수집 완료 → {RAW_PATH}")
    if failed_channels:
        print(f"실패한 채널 ({len(failed_channels)}개): {failed_channels}")


## 1-2. 채널 확장 (스노우볼 샘플링)

수집된 텍스트에서 `t.me/...` 형태의 채널 링크를 추출해 크롤링 대상을 넓힙니다.


In [ ]:
import re

TME_PATTERN = re.compile(r"t\.me/(joinchat/[\w-]+|\+[\w-]+|[\w_]{5,})")

def extract_new_channels(text):
    return TME_PATTERN.findall(text or "")

if not DEMO_MODE:
    discovered = set()
    with open(RAW_PATH, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            discovered.update(extract_new_channels(rec["text"]))
    new_channels = discovered - set(SEED_CHANNELS)
    print(f"신규 발견 채널 수: {len(new_channels)}")
    # 필요 시 new_channels를 SEED_CHANNELS에 추가해 crawl_channel을 재실행하세요.


## 1-3. 데이터 로드 및 전처리

`DEMO_MODE`에 따라 실제 수집 결과(`telegram_raw.jsonl`) 또는 내장 샘플 데이터를 불러와
정규화 + "추천/시그널 성격" 키워드 매칭을 수행합니다. (도메인 WHOIS·문장 임베딩 등 리딩방 자체의
스팸 여부 판별용 피처는 새 파이프라인의 목적과 맞지 않아 제거했습니다.)


In [ ]:
import pandas as pd

SAMPLE_DATA = [
    # ── 리딩방 스팸으로 의심되는 패턴 (여러 채널에서 유사 문구 반복) ──
    {"channel": "stock_free_1", "channel_id": 1001, "message_id": 1, "date": "2026-07-01T09:00:00",
     "sender_id": 5001, "text": "무료 종목 리딩 받아가세요! 이번주 수익률 인증 92% 선착순 카톡 오픈채팅 open.kakao.com/o/gAbC123",
     "views": 15000, "forwards": 320, "fwd_from": None},
    {"channel": "stock_free_2", "channel_id": 1002, "message_id": 1, "date": "2026-07-01T09:05:00",
     "sender_id": 5002, "text": "무료 종목 리딩방 오픈! 이번주 수익 인증 92% 선착순 마감 임박 open.kakao.com/o/gAbC123",
     "views": 14800, "forwards": 305, "fwd_from": None},
    {"channel": "stock_free_3", "channel_id": 1003, "message_id": 1, "date": "2026-07-01T09:10:00",
     "sender_id": 5003, "text": "급등주 무료 리딩 단톡방 초대 수익인증 92% 선착순 open.kakao.com/o/gAbC123",
     "views": 15200, "forwards": 340, "fwd_from": None},
    {"channel": "stock_free_1", "channel_id": 1001, "message_id": 2, "date": "2026-07-02T10:00:00",
     "sender_id": 5001, "text": "타점 잡아드립니다 무료 종목 리딩 단톡방 https://t.me/+xYzAbCd",
     "views": 9000, "forwards": 210, "fwd_from": None},
    {"channel": "stock_free_4", "channel_id": 1004, "message_id": 1, "date": "2026-07-03T11:00:00",
     "sender_id": 5004, "text": "선착순 무료 리딩방 수익 인증 92% open.kakao.com/o/gAbC123 서두르세요",
     "views": 16000, "forwards": 360, "fwd_from": None},
    # ── 실제 종목명을 언급하는 "추천성" 메시지 (2단계 종목 추출 데모용) ──
    {"channel": "stock_free_1", "channel_id": 1001, "message_id": 3, "date": "2026-07-05T09:30:00",
     "sender_id": 5001, "text": "현대약품 오늘 상한가 갑니다! 지금 안 사면 후회함, 무료 리딩 신청받아요 open.kakao.com/o/gAbC123",
     "views": 12000, "forwards": 250, "fwd_from": None},
    {"channel": "stock_free_4", "channel_id": 1004, "message_id": 2, "date": "2026-07-05T10:00:00",
     "sender_id": 5004, "text": "SK하이닉스 급등 시작! 지금 진입 타이밍입니다 단톡방 링크 open.kakao.com/o/gAbC123",
     "views": 13000, "forwards": 270, "fwd_from": None},
    {"channel": "CryptoRich33", "channel_id": 1005, "message_id": 1, "date": "2026-07-06T08:00:00",
     "sender_id": 5005, "text": "이더리움 단기 저점 진입 완료 20% 수익 무료 선물 시그널 신청 https://forms.gle/a8s7wVoQ2DeJiVUv7",
     "views": 8000, "forwards": 150, "fwd_from": None},
    # ── 정상적인 일반 대화/뉴스 공유로 추정되는 메시지 ──
    {"channel": "econ_news_kr", "channel_id": 2001, "message_id": 1, "date": "2026-07-01T08:00:00",
     "sender_id": 6001, "text": "오늘 코스피 지수는 전일 대비 0.4% 상승 마감했습니다. 외국인 순매수 전환.",
     "views": 500, "forwards": 3, "fwd_from": None},
    {"channel": "econ_news_kr", "channel_id": 2001, "message_id": 2, "date": "2026-07-02T08:00:00",
     "sender_id": 6001, "text": "금일 금통위 기준금리 동결 발표, 시장 예상과 부합.",
     "views": 480, "forwards": 2, "fwd_from": None},
    {"channel": "econ_news_kr", "channel_id": 2001, "message_id": 3, "date": "2026-07-06T08:00:00",
     "sender_id": 6001, "text": "삼성전자 3분기 실적 발표, 시장 예상치에 부합하는 흐름을 보였습니다.",
     "views": 510, "forwards": 4, "fwd_from": None},
    {"channel": "invest_study_group", "channel_id": 2002, "message_id": 1, "date": "2026-07-02T14:00:00",
     "sender_id": 6002, "text": "재무제표 분석 스터디 이번주 토요일 오후 2시에 진행합니다. 참여 원하시는 분은 댓글 남겨주세요.",
     "views": 120, "forwards": 1, "fwd_from": None},
    {"channel": "invest_study_group", "channel_id": 2002, "message_id": 2, "date": "2026-07-03T14:00:00",
     "sender_id": 6003, "text": "지난주 스터디 자료 공유드립니다. 링크는 곧 올리겠습니다.",
     "views": 110, "forwards": 0, "fwd_from": None},
    {"channel": "personal_diary_ch", "channel_id": 2003, "message_id": 1, "date": "2026-07-04T20:00:00",
     "sender_id": 6004, "text": "오늘 하루도 고생 많으셨습니다. 내일은 더 좋은 하루가 되길 바랍니다.",
     "views": 40, "forwards": 0, "fwd_from": None},
]

if DEMO_MODE:
    df = pd.DataFrame(SAMPLE_DATA)
else:
    rows = []
    with open(RAW_PATH, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)

# 실제 크롤링 데이터에는 미디어만 있고 텍스트가 없는 메시지(NaN)가 섞여 있을 수 있어 공백으로 채움
df["text"] = df["text"].fillna("")

print(f"로드된 메시지 수: {len(df)}")
df.head()


### 크롤링 원본 데이터 다운로드 (선택)

이후 단계에 들어가기 전에, 지금까지 불러온 원본 메시지 데이터를 먼저 CSV/XLSX로
저장하고 다운로드하고 싶다면 아래 셀을 실행하세요.


In [ ]:
RAW_CSV_PATH = "telegram_raw_messages.csv"
RAW_XLSX_PATH = "telegram_raw_messages.xlsx"

df.to_csv(RAW_CSV_PATH, index=False, encoding="utf-8-sig")
df.to_excel(RAW_XLSX_PATH, index=False)
print(f"저장 완료 → {RAW_CSV_PATH}, {RAW_XLSX_PATH} ({len(df)}건)")

try:
    from google.colab import files
    files.download(RAW_CSV_PATH)
    files.download(RAW_XLSX_PATH)
except ImportError:
    print("Colab 환경이 아니므로 자동 다운로드는 건너뜁니다. 파일 탐색기에서 직접 받으세요.")


In [ ]:
KEYWORD_PATTERNS = [
    r"무료\s*(리딩|종목)", r"수익\s*인증", r"선착순", r"\d+%\s*수익",
    r"단톡방", r"카톡\s*오픈채팅", r"급등주", r"타점",
]

def normalize(text):
    text = re.sub(r"[\u200b\uFE0F\u3164]", "", text)  # 제로폭/변형 문자 제거
    return text.strip()

def keyword_hits(text):
    return sum(bool(re.search(p, text or "")) for p in KEYWORD_PATTERNS)

df["text_norm"] = df["text"].map(normalize)
df["kw_hits"] = df["text_norm"].map(keyword_hits)  # "추천/시그널 성격"이 강한 메시지일수록 높음

df[["channel", "text_norm", "kw_hits"]]


---
# 2단계. 리딩방 메시지에서 추천 종목/기업 추출

간단한 사전(dictionary) 매칭 방식으로 메시지에서 종목명을 추출합니다. 실전에서는 KRX 상장 종목 전체 리스트로
사전을 확장하거나, 한국어 개체명 인식(NER) 모델로 대체할 수 있습니다. 여기서는 데모로 소수 종목만 등록합니다.


In [ ]:
STOCK_DICTIONARY = [
    {"name": "현대약품", "ticker": "004310", "asset_type": "주식"},
    {"name": "SK하이닉스", "ticker": "000660", "asset_type": "주식"},
    {"name": "삼성전자", "ticker": "005930", "asset_type": "주식"},
    {"name": "이더리움", "ticker": None, "asset_type": "코인"},
    {"name": "비트코인", "ticker": None, "asset_type": "코인"},
]

def extract_recommended_stocks(text):
    text = text or ""
    return [item["name"] for item in STOCK_DICTIONARY if item["name"] in text]

df["recommended_stocks"] = df["text_norm"].map(extract_recommended_stocks)

# 메시지 단위 → (채널, 메시지, 종목) 단위로 펼치기
mention_rows = []
for _, row in df.iterrows():
    for stock in row["recommended_stocks"]:
        mention_rows.append({
            "channel": row["channel"],
            "message_id": row["message_id"],
            "date": row["date"],
            "stock_name": stock,
            "kw_hits": row["kw_hits"],
            "text_norm": row["text_norm"],
        })

mentions_df = pd.DataFrame(mention_rows)
print(f"추출된 종목 언급 수: {len(mentions_df)}건 (메시지 {len(df)}건 중)")

if len(mentions_df):
    mention_summary = (
        mentions_df.groupby("stock_name")
        .agg(mention_count=("message_id", "count"), channel_count=("channel", "nunique"),
             first_mentioned=("date", "min"))
        .reset_index()
        .sort_values("mention_count", ascending=False)
    )
else:
    mention_summary = pd.DataFrame(columns=["stock_name", "mention_count", "channel_count", "first_mentioned"])

mention_summary


---
# 3단계. 추천 종목 관련 언론 기사 조사

네이버 뉴스 검색 API(`https://openapi.naver.com/v1/search/news.json`)로 각 추천 종목의 최근 기사를 조회합니다.
Client ID/Secret은 [네이버 개발자센터](https://developers.naver.com/apps/#/register)에서 애플리케이션 등록 후 발급받습니다.


In [ ]:
import requests
from urllib.parse import quote

SAMPLE_NEWS = [
    # ── "현대약품" 관련 — 동시다발적 보도자료성 기사(작전 의심 패턴) ──
    {"stock_name": "현대약품", "title": "현대약품, 특징주 부각...단기 급등세",
     "description": "현대약품이 특징주로 부각되며 단기 급등세를 보이고 있다. 매수 문의가 폭주하는 모습이다.",
     "pubDate": "2026-07-05T11:00:00", "link": "https://press-release-a.example.com/1"},
    {"stock_name": "현대약품", "title": "현대약품 특징주 부각, 단기 급등세 지속",
     "description": "현대약품이 특징주로 부각되며 단기 급등세를 지속하고 있다. 매수 문의가 폭주하는 상황.",
     "pubDate": "2026-07-05T11:30:00", "link": "https://press-release-b.example.com/1"},
    {"stock_name": "현대약품", "title": "[특징주] 현대약품, 급등세... 매수 문의 폭주",
     "description": "현대약품이 특징주로 부각되며 단기 급등세를 보이는 중이다. 매수 문의가 폭주하고 있다.",
     "pubDate": "2026-07-05T12:00:00", "link": "https://press-release-c.example.com/1"},
    # ── "SK하이닉스" 관련 — 정상적인 실적/업황 기사 ──
    {"stock_name": "SK하이닉스", "title": "SK하이닉스, HBM 수요 확대에 3분기 실적 개선 전망",
     "description": "증권가는 SK하이닉스의 HBM 수요 확대로 3분기 실적이 시장 예상치를 상회할 것으로 내다봤다.",
     "pubDate": "2026-07-05T09:00:00", "link": "https://real-news-outlet.example.com/2"},
    # ── "삼성전자" 관련 — 정상적인 실적 기사 ──
    {"stock_name": "삼성전자", "title": "삼성전자 3분기 실적 예상치 부합, 반도체 업황 개선세",
     "description": "삼성전자가 3분기 실적을 발표하며 시장 예상치에 부합하는 흐름을 보였다.",
     "pubDate": "2026-07-06T09:00:00", "link": "https://real-news-outlet.example.com/3"},
]

def fetch_naver_news(query, display=20, client_id=None, client_secret=None):
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {"X-Naver-Client-Id": client_id, "X-Naver-Client-Secret": client_secret}
    params = {"query": query, "display": display, "sort": "date"}
    resp = requests.get(url, headers=headers, params=params, timeout=10)
    resp.raise_for_status()
    items = resp.json().get("items", [])
    return [
        {
            "stock_name": query,
            "title": re.sub("<.*?>", "", it["title"]),
            "description": re.sub("<.*?>", "", it["description"]),
            "pubDate": it["pubDate"],
            "link": it["link"],
        }
        for it in items
    ]

if not DEMO_MODE:
    naver_client_id = getpass("네이버 뉴스 API Client ID: ")
    naver_client_secret = getpass("네이버 뉴스 API Client Secret: ")

    news_rows = []
    for stock in mention_summary["stock_name"]:
        try:
            news_rows.extend(fetch_naver_news(stock, client_id=naver_client_id, client_secret=naver_client_secret))
        except Exception as e:
            print(f"  ✗ {stock} 뉴스 조회 실패: {type(e).__name__} — {e}")
    news_df = pd.DataFrame(news_rows)
else:
    print("DEMO_MODE=True → 실제 뉴스 API 호출을 건너뛰고 샘플 기사 데이터를 사용합니다.")
    news_df = pd.DataFrame(SAMPLE_NEWS)

news_df["press_domain"] = news_df["link"].map(lambda l: re.search(r"https?://([\w.-]+)", l).group(1) if l else None)
print(f"수집된 기사 수: {len(news_df)}건")
news_df


## 3-1. 언론사 신뢰도 분류
### (회의 반영: 언론사 신뢰도 스코어)

회의에서 나온 핵심 지적 — "리딩방에서 5만 원이면 네이버 뉴스에 기사를 뿌릴 수 있다" — 을 반영해,
기사를 낸 매체가 **주요 언론사**인지 **소규모 인터넷 언론사**인지를 구분합니다. 실전에서는 한국언론진흥재단의
매체 등록 현황이나 자체 화이트리스트로 확장해야 하며, 여기서는 데모로 소수 매체만 분류합니다.


In [ ]:
PRESS_TIER = {
    "real-news-outlet.example.com": {"tier": "주요언론사", "credibility": 1.0},
    "press-release-a.example.com": {"tier": "인터넷언론사(소규모)", "credibility": 0.2},
    "press-release-b.example.com": {"tier": "인터넷언론사(소규모)", "credibility": 0.2},
    "press-release-c.example.com": {"tier": "인터넷언론사(소규모)", "credibility": 0.2},
}
# 화이트리스트에 없는 매체는 "5만 원이면 기사를 뿌릴 수 있는" 무명 매체일 가능성을 고려해 보수적으로 소규모 취급
DEFAULT_PRESS_TIER = {"tier": "미분류(소규모로 간주)", "credibility": 0.3}

def classify_press(domain):
    return PRESS_TIER.get(domain, DEFAULT_PRESS_TIER)

news_df["press_tier"] = news_df["press_domain"].map(lambda d: classify_press(d)["tier"])
news_df["press_credibility"] = news_df["press_domain"].map(lambda d: classify_press(d)["credibility"])

news_df[["stock_name", "press_domain", "press_tier", "press_credibility"]]


## 3-2. 뉴스-주가 상관관계 확인
### (회의 반영: 뉴스-주가 상관관계)

뉴스가 몰렸다고 해서 전부 의심스러운 건 아닙니다. **실제 공시(실적·계약·임상 등)와 맞물려 있다면 정상적인 호재**이고,
공시 없이 뉴스만으로 가격·거래량이 흔들렸다면 세력의 "숫자놀음"에 가깝습니다. 실전에서는 DART(전자공시시스템)
OpenAPI로 같은 기간의 공시 유무를 조회해야 하며, 여기서는 데모 데이터로 대체합니다.


In [ ]:
SAMPLE_DISCLOSURES = {
    "현대약품": False,   # 뉴스 버스트 기간에 매칭되는 공식 공시 없음 → 뉴스만으로 만든 움직임 의심
    "SK하이닉스": True,  # 실적 관련 공시 존재
    "삼성전자": True,
}

disclosure_check = pd.DataFrame({
    "stock_name": list(SAMPLE_DISCLOSURES.keys()),
    "has_official_disclosure": list(SAMPLE_DISCLOSURES.values()),
})
disclosure_check


---
# 4단계. 뉴스 버스트·근접중복 탐지로 "작전세력 의심 보도" 1차 판별
### (이상치 탐지 적용 지점 ①)

정상적인 기업 뉴스라면 매체마다 문구가 다르고 실적·공시 같은 사실관계를 전달합니다. 반면 리딩방發 펌핑을
뒷받침하려는 보도자료성 기사는 **여러 매체가 짧은 기간에 거의 동일한 문구를 반복 게시**하는 경향이 있습니다.
이 "근접중복 반복성"은 원래 리딩방 스팸 탐지에 쓰던 MinHash 기법을 그대로 재사용해 잡아냅니다.


In [ ]:
from datasketch import MinHash, MinHashLSH

PUMP_PR_KEYWORDS = [r"특징주", r"급등세", r"매수\s*문의\s*폭주", r"단기\s*급등", r"테마\s*부각"]

def get_minhash(text, num_perm=64, shingle_size=4):
    # 한국어는 조사·어미가 붙어 공백 기준 단어 분리가 불안정하므로,
    # 공백을 제거한 문자 단위 n-gram(shingle)으로 유사도를 비교합니다.
    text = "".join(text.split())
    m = MinHash(num_perm=num_perm)
    shingles = {text[i:i + shingle_size] for i in range(max(len(text) - shingle_size + 1, 1))}
    for sh in shingles:
        m.update(sh.encode("utf8"))
    return m

def pr_keyword_hits(text):
    return sum(bool(re.search(p, text or "")) for p in PUMP_PR_KEYWORDS)

news_df["combined_text"] = (news_df["title"].fillna("") + " " + news_df["description"].fillna(""))
news_df["pr_keyword_hits"] = news_df["combined_text"].map(pr_keyword_hits)

lsh = MinHashLSH(threshold=0.3, num_perm=64)
duplicate_count = []
for idx, text in enumerate(news_df["combined_text"]):
    mh = get_minhash(text)
    matches = lsh.query(mh)
    duplicate_count.append(len(matches))
    lsh.insert(str(idx), mh)
news_df["duplicate_count"] = duplicate_count

# 리딩방에서 해당 종목이 처음 언급된 시점 대비, N일 이내 몰린 기사 수(버스트) 계산
BURST_WINDOW_DAYS = 3
first_mention = mention_summary.set_index("stock_name")["first_mentioned"] if len(mention_summary) else pd.Series(dtype="object")

def days_since_mention(row):
    if row["stock_name"] not in first_mention.index:
        return None
    mention_date = pd.to_datetime(first_mention[row["stock_name"]])
    news_date = pd.to_datetime(row["pubDate"])
    return (news_date - mention_date).total_seconds() / 86400

news_df["days_since_mention"] = news_df.apply(days_since_mention, axis=1)
news_df["is_burst"] = news_df["days_since_mention"].between(0, BURST_WINDOW_DAYS)

def normalize01(series):
    s = series.astype(float)
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng else s * 0

stock_news = news_df.groupby("stock_name").agg(
    article_count=("title", "count"),
    avg_duplicate_count=("duplicate_count", "mean"),
    total_pr_keyword_hits=("pr_keyword_hits", "sum"),
    burst_count=("is_burst", "sum"),
    minor_press_ratio=("press_credibility", lambda s: (s < 0.5).mean()),
).reset_index()

# 회의 반영: 공시 없이 뉴스만으로 몰린 경우("뉴스-주가 상관관계" 미스매치)를 별도 신호로 포함
stock_news = stock_news.merge(disclosure_check, on="stock_name", how="left")
stock_news["has_official_disclosure"] = stock_news["has_official_disclosure"].fillna(False)
stock_news["news_price_mismatch"] = (stock_news["burst_count"] > 0) & (~stock_news["has_official_disclosure"])

stock_news["pump_news_score"] = (
    0.25 * normalize01(stock_news["avg_duplicate_count"])
    + 0.15 * normalize01(stock_news["total_pr_keyword_hits"])
    + 0.2 * normalize01(stock_news["burst_count"])
    + 0.25 * stock_news["minor_press_ratio"]
    + 0.15 * stock_news["news_price_mismatch"].astype(float)
)

stock_news.sort_values("pump_news_score", ascending=False)[[
    "stock_name", "article_count", "avg_duplicate_count", "total_pr_keyword_hits",
    "burst_count", "minor_press_ratio", "has_official_disclosure", "news_price_mismatch", "pump_news_score",
]]


---
# 5단계. 기업 건전성 스코어링

재무/공시 지표로 "이 회사가 안정적인 회사인지"를 스코어링합니다. 실전에서는 DART(전자공시시스템) OpenAPI나
KRX 상장공시시스템에서 부채비율·감사의견·관리종목 지정 여부·최대주주 지분율 변동 등을 가져와야 하지만,
여기서는 데모용 샘플 재무 데이터를 사용합니다.


In [ ]:
# 데모용 샘플 재무/공시 데이터 (실제 수치 아님) — 실전에서는 DART OpenAPI 등으로 대체
SAMPLE_FUNDAMENTALS = [
    {"stock_name": "현대약품", "sector": "바이오", "market_cap_billion": 180, "debt_ratio": 145.0,
     "is_administrative_issue": 0, "audit_opinion": "한정", "major_holder_change_pct": -8.5},
    {"stock_name": "SK하이닉스", "sector": "반도체", "market_cap_billion": 135000, "debt_ratio": 38.0,
     "is_administrative_issue": 0, "audit_opinion": "적정", "major_holder_change_pct": 0.1},
    {"stock_name": "삼성전자", "sector": "반도체", "market_cap_billion": 400000, "debt_ratio": 27.0,
     "is_administrative_issue": 0, "audit_opinion": "적정", "major_holder_change_pct": 0.0},
]

fundamentals_df = pd.DataFrame(SAMPLE_FUNDAMENTALS)

# 부채비율이 높을수록, 관리종목일수록, 감사의견이 '적정'이 아닐수록, 최대주주 지분이 급변할수록 불안정 → 감점
fundamentals_df["debt_risk"] = normalize01(fundamentals_df["debt_ratio"])
fundamentals_df["audit_risk"] = (fundamentals_df["audit_opinion"] != "적정").astype(float)
fundamentals_df["holder_change_risk"] = normalize01(fundamentals_df["major_holder_change_pct"].abs())

fundamentals_df["stability_score"] = 1 - (
    0.4 * fundamentals_df["debt_risk"]
    + 0.3 * fundamentals_df["is_administrative_issue"]
    + 0.2 * fundamentals_df["audit_risk"]
    + 0.1 * fundamentals_df["holder_change_risk"]
)

# 회의 반영: 소액으로도 주가가 흔들리는 "취약 기업" 특성 — 시가총액이 작을수록, 바이오 섹터일수록 소액 개입에 취약
fundamentals_df["small_cap_risk"] = normalize01(1 / fundamentals_df["market_cap_billion"])
fundamentals_df["sector_risk"] = (fundamentals_df["sector"] == "바이오").astype(float)
fundamentals_df["vulnerability_score"] = (
    0.6 * fundamentals_df["small_cap_risk"] + 0.4 * fundamentals_df["sector_risk"]
)

fundamentals_df[[
    "stock_name", "sector", "market_cap_billion", "debt_ratio", "is_administrative_issue",
    "audit_opinion", "stability_score", "vulnerability_score",
]]


---
# 6단계. 거래량 급등 탐지 + 종합 판단 (건전 투자 유도 북극성 지표)
### (이상치 탐지 적용 지점 ②)

"거래량이 평소 대비 300% 이상 급등"하는 현상 자체가 고전적인 이상치 탐지 문제입니다. 20일 이동평균 대비
당일 거래량 배율(z-score 개념)로 급등을 잡아내고, 보조적으로 Isolation Forest로도 교차 검증합니다.


In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

# 데모용 샘플 일별 거래량/가격 데이터 (실제로는 KRX/증권사 API 등에서 수집)
np.random.seed(42)
volume_rows = []
for stock in fundamentals_df["stock_name"]:
    base_volume = np.random.randint(500_000, 2_000_000)
    for day in range(20):
        volume = int(base_volume * np.random.uniform(0.8, 1.2))
        price_change_pct = np.random.uniform(-2, 2)
        volume_rows.append({"stock_name": stock, "day": day, "volume": volume, "price_change_pct": price_change_pct})
    # 마지막 날, "현대약품"만 인위적으로 거래량 급등(리딩방 추천 직후 상황을 재현)
    if stock == "현대약품":
        volume_rows[-1]["volume"] = int(base_volume * 4.2)
        volume_rows[-1]["price_change_pct"] = 18.5

volume_df = pd.DataFrame(volume_rows)

def compute_volume_ratio(group):
    group = group.sort_values("day").copy()
    rolling_avg = group["volume"].rolling(window=19, min_periods=5).mean().shift(1)
    group["volume_ratio"] = group["volume"] / rolling_avg
    return group

volume_df = pd.concat(
    [compute_volume_ratio(g) for _, g in volume_df.groupby("stock_name")],
    ignore_index=True,
)
volume_df["volume_spike_flag"] = volume_df["volume_ratio"] >= 3.0  # 300% 이상 급등

# Isolation Forest로 [거래량 배율, 가격 변동률] 조합 자체의 통계적 이상 여부도 교차 확인
iso_input = volume_df.dropna(subset=["volume_ratio"])[["volume_ratio", "price_change_pct"]]
if len(iso_input) >= 5:
    iso_clf = IsolationForest(n_estimators=200, contamination=0.1, random_state=42)
    iso_clf.fit(iso_input)
    volume_df.loc[iso_input.index, "iso_anomaly_score"] = -iso_clf.score_samples(iso_input)

latest_volume = volume_df.sort_values("day").groupby("stock_name").tail(1)
latest_volume[["stock_name", "day", "volume", "volume_ratio", "volume_spike_flag", "iso_anomaly_score"]]


## 종합 판단: 건전 투자 유도 라벨링

종목추출(2단계) · 언론사 신뢰도·뉴스-주가 상관관계(3단계) · 뉴스 버스트(4단계) · 기업 건전성·취약도(5단계) ·
거래량 급등(6단계)을 모두 합쳐, "묻지마 추종매수 위험"과 "정상적인 호재 가능성"을 구분하는 최종 표를 만듭니다.
소형주·바이오 기업처럼 소액 개입에 취약한 기업은 더 낮은 의심 점수에서도 경고가 뜨도록 임계값을 동적으로 낮춥니다.


In [ ]:
final_df = (
    mention_summary
    .merge(stock_news[["stock_name", "pump_news_score", "news_price_mismatch"]], on="stock_name", how="left")
    .merge(fundamentals_df[["stock_name", "sector", "stability_score", "vulnerability_score"]], on="stock_name", how="left")
    .merge(latest_volume[["stock_name", "volume_ratio", "volume_spike_flag"]], on="stock_name", how="left")
)

final_df["pump_news_score"] = final_df["pump_news_score"].fillna(0)
final_df["news_price_mismatch"] = final_df["news_price_mismatch"].fillna(False)
final_df["stability_score"] = final_df["stability_score"].fillna(0.5)
final_df["vulnerability_score"] = final_df["vulnerability_score"].fillna(0.0)
final_df["volume_spike_flag"] = final_df["volume_spike_flag"].fillna(False)

# 회의 반영: 소액 개입에 취약한(소형주·바이오) 기업일수록, 더 낮은 pump_news_score에서도 경고가 뜨도록
# 임계값 자체를 낮춥니다 (취약도 1.0이면 임계값 0.5 → 0.3까지 낮아짐).
final_df["pump_threshold"] = 0.5 - 0.2 * final_df["vulnerability_score"]

def classify(row):
    if row["volume_spike_flag"] and row["pump_news_score"] >= row["pump_threshold"] and row["stability_score"] < 0.5:
        return "⚠ 투자 주의 (묻지마 추종매수 위험)"
    if row["volume_spike_flag"] and row["stability_score"] >= 0.5 and row["pump_news_score"] < row["pump_threshold"]:
        return "정상 호재 가능성 (모니터링)"
    return "특이사항 없음"

final_df["investment_guidance"] = final_df.apply(classify, axis=1)

final_df[[
    "stock_name", "sector", "mention_count", "pump_news_score", "news_price_mismatch",
    "stability_score", "vulnerability_score", "pump_threshold",
    "volume_ratio", "volume_spike_flag", "investment_guidance"
]]


### 북극성 지표 계산 방식 (건전 투자 유도율)

> **북극성 지표**: 거래량 급등이 발생한 종목 중, 사전에 "투자 주의" 경고가 노출된 비율.
>
> `건전 투자 유도율 = (급등 + 주의 라벨이 붙은 종목 수) / (전체 거래량 급등 종목 수) × 100`
>
> 실제 서비스에서는 여기에 "경고 노출 이후 실제 매수 억제 효과"(예: 경고 노출 전후 거래량 변화)까지 연결해야
> 진짜 북극성이 완성되지만, 이 노트북에서는 종목 자료만으로 계산 가능한 부분까지만 구현했습니다.


In [ ]:
spiked = final_df[final_df["volume_spike_flag"]]
if len(spiked):
    healthy_guidance_rate = (spiked["investment_guidance"] == "⚠ 투자 주의 (묻지마 추종매수 위험)").mean() * 100
    print(f"거래량 급등 종목 {len(spiked)}개 중 '투자 주의' 경고 비율(건전 투자 유도율): {healthy_guidance_rate:.1f}%")
else:
    print("이번 데이터에서는 거래량 급등(300%+)이 감지된 종목이 없습니다.")


---
# 7단계. 결과 저장 및 다운로드


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(final_df["stock_name"], final_df["pump_news_score"], label="pump_news_score")
plt.bar(final_df["stock_name"], final_df["stability_score"], alpha=0.6, label="stability_score")
plt.title("종목별 뉴스 의심도 vs 기업 안정성 점수")
plt.ylabel("score")
plt.legend()
plt.tight_layout()
plt.show()

OUT_CSV_PATH = "investment_guidance_result.csv"
OUT_XLSX_PATH = "investment_guidance_result.xlsx"
final_df.to_csv(OUT_CSV_PATH, index=False, encoding="utf-8-sig")
final_df.to_excel(OUT_XLSX_PATH, index=False)
print(f"결과 저장 완료 → {OUT_CSV_PATH}, {OUT_XLSX_PATH}")

try:
    from google.colab import files
    files.download(OUT_CSV_PATH)
    files.download(OUT_XLSX_PATH)
except ImportError:
    print("Colab 환경이 아니므로 자동 다운로드는 건너뜁니다. 파일 탐색기에서 직접 받으세요.")


---
## 한계 및 유의사항

- `DEMO_MODE`의 샘플 데이터(뉴스, 재무, 거래량)는 전부 파이프라인 동작 확인용 예시이며 실제 수치가 아닙니다.
- 2단계 종목 추출은 소수 종목만 등록한 사전(dictionary) 매칭 방식입니다. 실전에서는 KRX 전체 상장 종목 리스트로
  사전을 확장하거나 한국어 개체명 인식(NER) 모델로 교체해야 오탐/누락을 줄일 수 있습니다.
- 3단계 네이버 뉴스 검색 API는 호출량 제한이 있고, 언론사(press) 정보를 명시적으로 제공하지 않아 URL 도메인으로
  추정합니다. 매체명이 필요하면 별도 매체 데이터베이스와 매칭해야 합니다.
- 4단계 "작전세력 의심 보도" 판별은 근접중복·버스트 타이밍 등 **통계적 의심 신호**일 뿐입니다. 실제로 허위·과장
  보도인지는 기사 원문 검증과 사람의 판단이 반드시 필요하며, 이 노트북의 결과만으로 특정 매체·기사를 단정해서는
  안 됩니다.
- 5단계 기업 건전성 스코어는 데모 재무 지표 4가지만 반영한 단순 가중합입니다. 실전에서는 DART OpenAPI 등에서
  더 많은 공시 지표를 가져와 스코어카드를 정교화해야 합니다.
- 6단계 거래량 급등 탐지는 20일 이동평균 대비 배율(z-score 유사 개념)과 Isolation Forest를 함께 썼습니다.
  실전 데이터는 결측·상장폐지·거래정지 등으로 더 복잡하므로 추가 전처리가 필요합니다.
- 3-1단계 언론사 신뢰도 분류는 화이트리스트 4개 도메인만 등록한 데모입니다. 실전에서는 한국언론진흥재단의 매체
  등록 현황 등으로 화이트리스트를 크게 확장해야 하며, 미등록 매체를 전부 "소규모"로 간주하는 보수적 규칙도
  오탐 가능성이 있어 주기적으로 재검토해야 합니다.
- 3-2단계 공시 매칭 여부(`SAMPLE_DISCLOSURES`)는 데모 값입니다. 실전에서는 DART OpenAPI로 종목별·기간별 실제
  공시 유무를 조회해야 합니다.
- 5단계 취약도(`vulnerability_score`)와 6단계의 동적 임계값 계산식(0.5 − 0.2×취약도)은 회의 논의를 반영한
  초기 설계일 뿐 검증된 가중치가 아닙니다. 실제 사례로 임계값과 가중치를 다시 튜닝해야 합니다.
- 북극성 지표("건전 투자 유도율")는 이 노트북에서 계산 가능한 부분(경고 노출 비율)까지만 구현했습니다. 실제
  "묻지마 매수 억제 효과"까지 검증하려면 경고 노출 전후의 실제 거래 데이터가 추가로 필요합니다.
